In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pymupdf pytesseract
!pip install -q flask pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 44.5 MB/s eta 0:00:00


In [ ]:
import re
import json
import torch
import pymupdf
import pytesseract
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen3-4B loaded successfully.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-4B loaded successfully.


In [ ]:
def generate_video_blueprint(source_text):

    prompt = f"""
You are an AI system that converts a source document into a
structured video blueprint.

SOURCE DOCUMENT:
{source_text}

TASK:
Create a video blueprint using ONLY information explicitly
present in the source document.

IMPORTANT SOURCE-GROUNDING RULES:
1. Do NOT invent facts, statistics, examples, causes, effects,
   recommendations, or procedures.
2. Do NOT add information from your general knowledge.
3. Do NOT infer information that is not explicitly stated.
4. Preserve important technical terminology from the source.
5. Remove obvious OCR noise, page headers, page numbers,
   timestamps, and formatting artifacts.
6. If the source does not provide enough information for a detail,
   do not invent it.
7. Every narration statement must be supported by the source.

VIDEO REQUIREMENTS:
- Create exactly 5 scenes.
- Each scene should have a duration of 12 seconds.
- Narration should contain 2–3 concise sentences.
- Visual descriptions must correspond to information in the source.
- On-screen text must use only source-supported information.
- Audio cues must be simple and relevant.
- The complete narration should combine the narration from all scenes.
- Create subtitles from the generated narration.
- Subtitles must contain only the generated narration text.
- Do not add information that is not present in the source.

Return ONLY valid JSON.
Do not use markdown.
Do not include explanations outside the JSON.

JSON FORMAT:
{{
  "title": "...",
  "target_audience": "General Public",
  "duration_seconds": 60,

  "scenes": [
    {{
      "scene_number": 1,
      "duration_seconds": 12,
      "visual_description": "...",
      "on_screen_text": "...",
      "narration": "...",
      "audio_cue": "...",
      "visual_recommendation": "..."
    }},
    {{
      "scene_number": 2,
      "duration_seconds": 12,
      "visual_description": "...",
      "on_screen_text": "...",
      "narration": "...",
      "audio_cue": "...",
      "visual_recommendation": "..."
    }},
    {{
      "scene_number": 3,
      "duration_seconds": 12,
      "visual_description": "...",
      "on_screen_text": "...",
      "narration": "...",
      "audio_cue": "...",
      "visual_recommendation": "..."
    }},
    {{
      "scene_number": 4,
      "duration_seconds": 12,
      "visual_description": "...",
      "on_screen_text": "...",
      "narration": "...",
      "audio_cue": "...",
      "visual_recommendation": "..."
    }},
    {{
      "scene_number": 5,
      "duration_seconds": 12,
      "visual_description": "...",
      "on_screen_text": "...",
      "narration": "...",
      "audio_cue": "...",
      "visual_recommendation": "..."
    }}
  ],

  "full_narration": "...",

  "subtitles": [
    {{
      "scene_number": 1,
      "text": "..."
    }},
    {{
      "scene_number": 2,
      "text": "..."
    }},
    {{
      "scene_number": 3,
      "text": "..."
    }},
    {{
      "scene_number": 4,
      "text": "..."
    }},
    {{
      "scene_number": 5,
      "text": "..."
    }}
  ]
}}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=2200,
        do_sample=False
    )

    result_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    if result_text.startswith("```"):
        result_text = result_text.replace("```json", "")
        result_text = result_text.replace("```", "").strip()

    start = result_text.find("{")
    end = result_text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("Qwen did not return valid JSON.")

    result_text = result_text[start:end + 1]

    return json.loads(result_text)

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

app = Flask(__name__)

@app.route("/generate", methods=["POST"])
def generate():

    data = request.get_json()

    if not data or "text" not in data:
        return jsonify({
            "error": "Source text is required."
        }), 400

    try:
        blueprint = generate_video_blueprint(
            data["text"]
        )

        return jsonify(blueprint)

    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 500


def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


thread = threading.Thread(target=run_flask)
thread.start()

print("Flask API started on port 5000")

Flask API started on port 5000


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3I5bFWut4ZumymVBwgDVlvLQpqB_3J2GMMsXcNMZnoTGrFNKs")

public_url = ngrok.connect(5000)

print("Public API URL:")
print(public_url)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Public API URL:
NgrokTunnel: "https://disbelief-class-spotless.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import requests

test_url = "https://disbelief-class-spotless.ngrok-free.dev/generate"

test_data = {
    "text": "Ransomware is a type of malware that encrypts files or locks systems until a ransom is paid."
}

response = requests.post(
    test_url,
    json=test_data,
    timeout=300
)

print("Status code:", response.status_code)
print("Response:")
print(response.text[:2000])

INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 15:10:27] "POST /generate HTTP/1.1" 200 -


Status code: 200
Response:
{"duration_seconds":60,"full_narration":"Ransomware is a type of malware. It encrypts files or locks systems. Ransomware is designed to demand payment. Users are often presented with a ransom note. After payment, files are typically unlocked. However, there is no guarantee of data recovery.","scenes":[{"audio_cue":"Soft background music with a subtle computer sound effect.","duration_seconds":12,"narration":"Ransomware is a type of malware. It encrypts files or locks systems.","on_screen_text":"Ransomware encrypts files.","scene_number":1,"visual_description":"A computer screen showing a file being encrypted.","visual_recommendation":"Show a close-up of a locked file icon."},{"audio_cue":"Slight tension-building sound effect.","duration_seconds":12,"narration":"Ransomware is designed to demand payment. Users are often presented with a ransom note.","on_screen_text":"Ransomware demands payment.","scene_number":2,"visual_description":"A user receiving a ransom 